In [13]:
from jinja2 import optimizer
from torch.utils.data import DataLoader

"""
    控制图的记录
    - 默认状态下,pytorch 会尽量记账
    - no_grad():这段计算先别记
        1.require_grad 是张量的属性,表示这个张量是否有资格被记录到
        2. no_grad() 是上下文状态,表示当前的计算是需要被记录
    - enable_grad(): 局部重新打开记录
    - inference_mode() :这是纯推理实现
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch  version: ", torch.__version__)
model = nn.Linear(6, 4)
x = torch.randn(10, 6)
y = torch.randn(10, 4)

#默认状态下,pytorch 会尽量记账
y_pred = model(x)

print("y_pred.requires_grad: ", y_pred.requires_grad)
print("y_pred.grad_fn: ", y_pred.grad_fn.name())

for name, param in model.named_parameters():
    print(f"{name}.requires_grad : {param.requires_grad}")

loss = F.mse_loss(y_pred, y)
loss.backward()

assert model.weight.grad is not True
assert model.bias.grad is not None

# no_grad():这段计算先别记,在no_grad() 模式下,一旦一个张量不在追踪,后续基于它的计算也不会被追踪,

with  torch.no_grad():
    y_pred = model(x)

print("y_pred.requires_grad: ", y_pred.requires_grad)
print("y_pred.grad_fn: ", y_pred.grad_fn)

loss = F.mse_loss(y_pred, y)
print("loss.requires_grad: ", loss.requires_grad)

try:
    loss.backward()
except RuntimeError as err:
    print("RuntimeError: ", err)

model.eval()
#
# with  torch.no_grad():
#     for x,y in valid_loader:
#         y_pred =  model(x)
#         loss =  loss_fn(y_pred,y)


## no_grad() 不会修改张量的本身的requires_grad属性

x = torch.randn(10, 6, requires_grad=True)

with torch.no_grad():
    print("x.requires_grad: ", x.requires_grad)
    z = x.sin()
    print("z.requires_grad: ", z.requires_grad)

# enable_grad(): 局部重新打开记录

x = torch.randn(4, requires_grad=True)

with torch.no_grad():
    a = x.sin()
    print('a.requires_grad: ', a.requires_grad)

    with  torch.enable_grad():
        b = x.cos()
        print('b.requires_grad: ', b.requires_grad)
    c = x.tan()
    print('c.requires_grad: ', c.requires_grad)

is_training = True
x = torch.randn(10, 6)

with torch.set_grad_enabled(is_training):
    y_pred = model(x)

print("y_pred.requires_grad: ", y_pred.requires_grad)


# 公用同样验证和执行代码,使用参数化

def run_one_epoch(model: Module, dataloader: DataLoader, training: bool = True):
    model.train()

    with  torch.set_grad_enabled(training):
        for x, y in dataloader:
            y_pred = model(x)
            loss = loss_fn(y_pred, y)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()


# inference_mode()  纯推理实现,没有重新开启的可能

with torch.inference_mode():
    x = torch.randn(4)


try:
    x.requires_grad_()
except RuntimeError as err:
    print("RuntimeError: ", err)


PyTorch  version:  2.13.0+cpu
y_pred.requires_grad:  True
y_pred.grad_fn:  AddmmBackward0
weight.requires_grad : True
bias.requires_grad : True
y_pred.requires_grad:  False
y_pred.grad_fn:  None
loss.requires_grad:  False
RuntimeError:  element 0 of tensors does not require grad and does not have a grad_fn
x.requires_grad:  True
z.requires_grad:  False
a.requires_grad:  False
b.requires_grad:  True
c.requires_grad:  False
y_pred.requires_grad:  True
RuntimeError:  Setting requires_grad=True on inference tensor outside InferenceMode is not allowed.
